In [ ]:
# Parameters
city_key = "ywg"
baseline_feed_id = "2024-12-15"
comparison_feed_id = "current"
save_figures = False
figures_dir = "reports/pr2/figures"
dpi = 200

# PR2: Route Schedule Comparison

This notebook compares pre-PTN and current scheduled route metrics using **Jaccard stop-set similarity** for entity resolution. Routes are matched by shared stop infrastructure ($J = |S_\text{pre} \cap S_\text{post}| / |S_\text{pre} \cup S_\text{post}|$), supplemented by exact short-name matches. This replaces the tier-aggregate workaround from the initial PR2 draft.

## 1. Setup & Helpers

In [ ]:
from pathlib import Path
import warnings

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import TwoSlopeNorm
from shapely.geometry import LineString

warnings.filterwarnings("ignore")

from ptn_analysis.context.reporting import (
    ensure_report_dirs,
    save_placeholder_figure,
    save_report_figure,
)

ensure_report_dirs("pr2")
figure_output_directory = Path(figures_dir)
figure_output_directory.mkdir(parents=True, exist_ok=True)

from ptn_analysis import TransitContext
from ptn_analysis.analysis import PTN_TIER_COLORS, PTN_TIER_ORDER


def build_route_gdf(db, feed_id="current"):
    """Build a GeoDataFrame of route geometries from GTFS shapes."""
    shape_routes = db.query(
        """
        SELECT r.route_short_name, t.shape_id, COUNT(*) as trip_count
        FROM ywg_trips t
        JOIN ywg_routes r ON t.route_id = r.route_id AND t.feed_id = r.feed_id
        WHERE t.feed_id = :feed_id AND t.shape_id IS NOT NULL
        GROUP BY r.route_short_name, t.shape_id
        ORDER BY r.route_short_name, trip_count DESC
        """,
        {"feed_id": feed_id},
    )
    best_shapes = shape_routes.drop_duplicates(subset="route_short_name", keep="first")
    shape_ids = best_shapes["shape_id"].unique().tolist()

    shapes = db.query(
        f"""
        SELECT shape_id, shape_pt_lat, shape_pt_lon, shape_pt_sequence
        FROM ywg_shapes
        WHERE shape_id IN ({','.join(f"'{s}'" for s in shape_ids)})
        ORDER BY shape_id, shape_pt_sequence
        """
    )

    lines = {}
    for shape_id, group in shapes.groupby("shape_id"):
        coords = list(zip(group["shape_pt_lon"], group["shape_pt_lat"]))
        if len(coords) >= 2:
            lines[shape_id] = LineString(coords)

    best_shapes = best_shapes.copy()
    best_shapes["geometry"] = best_shapes["shape_id"].map(lines)
    best_shapes = best_shapes.dropna(subset=["geometry"])
    return gpd.GeoDataFrame(best_shapes, geometry="geometry", crs="EPSG:4326")

## 2. Route Evolution Matching via Jaccard Similarity

Stop-set Jaccard similarity identifies equivalent routes across PTN redesign without manual crosswalks. For each pre-PTN route, we compute its stop set and compare against every post-PTN route's stop set. Routes with $J > 0.3$ are candidate matches; exact short-name matches are preserved regardless of $J$.

In [ ]:
ctx = TransitContext.from_defaults(feed_id=comparison_feed_id)
frequency_analyzer = ctx.frequency()
coverage_analyzer = ctx.coverage()

from ptn_analysis.analysis import classify_ptn_tier

# Query stop sets per route for both feeds
pre_route_stops = ctx.working_db.query("""
    SELECT DISTINCT r.route_short_name, st.stop_id
    FROM ywg_stop_times st
    JOIN ywg_trips t ON st.trip_id = t.trip_id AND st.feed_id = t.feed_id
    JOIN ywg_routes r ON t.route_id = r.route_id AND t.feed_id = r.feed_id
    WHERE t.feed_id = :fid
""", {"fid": baseline_feed_id})

post_route_stops = ctx.working_db.query("""
    SELECT DISTINCT r.route_short_name, st.stop_id
    FROM ywg_stop_times st
    JOIN ywg_trips t ON st.trip_id = t.trip_id AND st.feed_id = t.feed_id
    JOIN ywg_routes r ON t.route_id = r.route_id AND t.feed_id = r.feed_id
    WHERE t.feed_id = :fid
""", {"fid": comparison_feed_id})

# Build stop sets per route
pre_sets = pre_route_stops.groupby("route_short_name")["stop_id"].apply(set).to_dict()
post_sets = post_route_stops.groupby("route_short_name")["stop_id"].apply(set).to_dict()

# Compute Jaccard similarity for all (pre, post) pairs
JACCARD_THRESHOLD = 0.3
jaccard_rows = []
for pre_name, pre_stops in pre_sets.items():
    for post_name, post_stops in post_sets.items():
        intersection = len(pre_stops & post_stops)
        union = len(pre_stops | post_stops)
        if union == 0:
            continue
        j = intersection / union
        if j >= JACCARD_THRESHOLD or pre_name == post_name:
            jaccard_rows.append({
                "pre_route": pre_name,
                "post_route": post_name,
                "jaccard": round(j, 3),
                "shared_stops": intersection,
                "total_stops": union,
                "exact_name": pre_name == post_name,
            })

jaccard_df = pd.DataFrame(jaccard_rows)
if not jaccard_df.empty:
    jaccard_df = jaccard_df.sort_values("jaccard", ascending=False)

# Classify evolution type
def classify_evolution(row):
    if row["exact_name"] and row["jaccard"] >= 0.5:
        return "Preserved"
    elif row["exact_name"]:
        return "Restructured (same name)"
    elif row["jaccard"] >= 0.5:
        return "Renamed"
    else:
        return "Partial overlap"

if not jaccard_df.empty:
    jaccard_df["evolution_type"] = jaccard_df.apply(classify_evolution, axis=1)

# Best match per pre-PTN route (highest Jaccard)
best_matches = jaccard_df.loc[jaccard_df.groupby("pre_route")["jaccard"].idxmax()] if not jaccard_df.empty else pd.DataFrame()

# Identify discontinued (pre-PTN routes with no match) and new (post-PTN only)
matched_pre = set(best_matches["pre_route"]) if not best_matches.empty else set()
matched_post = set(best_matches["post_route"]) if not best_matches.empty else set()
discontinued = sorted(set(pre_sets.keys()) - matched_pre)
new_routes = sorted(set(post_sets.keys()) - matched_post)

n_preserved = len(best_matches[best_matches["evolution_type"] == "Preserved"]) if not best_matches.empty else 0
n_renamed = len(best_matches[best_matches["evolution_type"].isin(["Renamed", "Partial overlap"])]) if not best_matches.empty else 0
print(f"Pre-PTN routes: {len(pre_sets)} | Post-PTN routes: {len(post_sets)}")
print(f"Preserved (same name, J>=0.5): {n_preserved}")
print(f"Jaccard-matched (renamed/restructured): {n_renamed}")
print(f"Discontinued: {len(discontinued)} | New: {len(new_routes)}")
display(best_matches.head(10) if not best_matches.empty else "No matches")

In [ ]:
# (Jaccard sankey deferred — baseline feed lacks stop-level data)


In [ ]:
# (Jaccard heatmap deferred — baseline feed lacks stop-level data)


In [ ]:
# Load route stats for both eras
pre_stats = ctx.working_db.query(
    "SELECT route_short_name, mean_headway, service_speed, num_trips "
    "FROM ywg_gtfs_route_stats WHERE feed_id = :fid",
    {"fid": baseline_feed_id},
)
post_stats = ctx.working_db.query(
    "SELECT route_short_name, mean_headway, service_speed, num_trips "
    "FROM ywg_gtfs_route_stats WHERE feed_id = :fid",
    {"fid": comparison_feed_id},
)

# Classify current routes by PTN tier
post_stats["ptn_tier"] = post_stats["route_short_name"].apply(lambda x: classify_ptn_tier(x)[0])

# Per-route averages across dates within each era
pre_route = pre_stats.groupby("route_short_name", as_index=False).agg(
    headway_pre=("mean_headway", "mean"),
    speed_pre=("service_speed", "mean"),
    trips_pre=("num_trips", "mean"),
)
post_route = post_stats.groupby(["route_short_name", "ptn_tier"], as_index=False).agg(
    headway_post=("mean_headway", "mean"),
    speed_post=("service_speed", "mean"),
    trips_post=("num_trips", "mean"),
)

# Build comparison using Jaccard matches + exact name matches
# 1. Exact name matches (inner join on route_short_name)
exact_matches = pre_route.merge(post_route, on="route_short_name", how="inner")

# 2. Jaccard-matched routes (renamed/restructured — pre_route != post_route)
jaccard_extra = pd.DataFrame()
if not best_matches.empty:
    renamed = best_matches[~best_matches["exact_name"]].copy()
    if not renamed.empty:
        jaccard_merged = renamed[["pre_route", "post_route", "jaccard", "evolution_type"]].merge(
            pre_route.rename(columns={"route_short_name": "pre_route"}),
            on="pre_route",
        ).merge(
            post_route.rename(columns={"route_short_name": "post_route"}),
            on="post_route",
        )
        jaccard_merged["route_short_name"] = jaccard_merged["pre_route"] + " -> " + jaccard_merged["post_route"]
        jaccard_extra = jaccard_merged[exact_matches.columns.tolist() + ["jaccard", "evolution_type"]] if "jaccard" in jaccard_merged.columns else jaccard_merged[exact_matches.columns]

# Combine: exact matches + Jaccard-matched renamed routes
route_comparison_table = exact_matches.copy()
if not jaccard_extra.empty:
    # Only add columns that exist in both
    common_cols = [c for c in exact_matches.columns if c in jaccard_extra.columns]
    route_comparison_table = pd.concat([exact_matches[common_cols], jaccard_extra[common_cols]], ignore_index=True)

route_comparison_table["headway_improvement"] = route_comparison_table["headway_pre"] - route_comparison_table["headway_post"]
route_comparison_table["speed_improvement"] = route_comparison_table["speed_post"] - route_comparison_table["speed_pre"]
route_comparison_table["trip_count_change"] = route_comparison_table["trips_post"] - route_comparison_table["trips_pre"]

# Jobs access comparison (neighbourhood-level, doesn't need route matching)
jobs_access_comparison_table = coverage_analyzer.jobs_access_comparison(
    baseline_feed_id=baseline_feed_id
)

n_pre = pre_route["route_short_name"].nunique()
n_post = post_route["route_short_name"].nunique()
n_exact = len(exact_matches)
n_jaccard = len(jaccard_extra) if not jaccard_extra.empty else 0
print(f"Pre-PTN routes: {n_pre} | Post-PTN routes: {n_post}")
print(f"Exact name matches: {n_exact} | Jaccard matches (renamed): {n_jaccard}")
print(f"Total comparison routes: {len(route_comparison_table)}")
print(f"Jobs access comparison rows: {len(jobs_access_comparison_table)}")
display(route_comparison_table.head())

## 3. Headway Improvement by PTN Tier

In [ ]:
def _best_label_point(geom, placed, candidates=None):
    """Pick the point along a LineString/MultiLineString farthest from already-placed labels."""
    from shapely.ops import nearest_points
    from shapely.geometry import Point, MultiPoint
    if candidates is None:
        candidates = [0.15, 0.25, 0.35, 0.5, 0.65, 0.75, 0.85]
    pts = []
    for f in candidates:
        try:
            pts.append(geom.interpolate(f, normalized=True))
        except Exception:
            pass
    if not pts:
        return geom.centroid
    if not placed:
        # First label: pick 0.35 (slightly off-center, away from downtown convergence)
        return pts[2] if len(pts) > 2 else pts[0]
    placed_mp = MultiPoint(placed)
    best_pt, best_dist = pts[0], 0
    for pt in pts:
        d = pt.distance(placed_mp)
        if d > best_dist:
            best_dist = d
            best_pt = pt
    return best_pt

if route_comparison_table.empty:
    save_placeholder_figure("prepost_combined.png", "Route comparison data is missing.",
                            figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
else:
    route_gdf = build_route_gdf(ctx.working_db, feed_id=comparison_feed_id)
    map_df = route_gdf.merge(
        route_comparison_table[["route_short_name", "headway_improvement", "speed_improvement",
                                 "trip_count_change", "ptn_tier"]],
        on="route_short_name", how="inner",
    ).to_crs(epsg=3857)
    neigh_gdf = ctx.working_db.neighbourhood_gdf().to_crs(epsg=3857)
    route_bounds = map_df.total_bounds
    x_pad = (route_bounds[2] - route_bounds[0]) * 0.08
    y_pad = (route_bounds[3] - route_bounds[1]) * 0.08

    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    panels = [
        ("headway_improvement", "Headway Change", "min", 6, 4),
        ("speed_improvement", "Speed Change", "km/h", 4, 3),
        ("trip_count_change", "Trip Count Change", "trips", 4, 3),
    ]
    for idx, (col, label, unit, n_best, n_worst) in enumerate(panels):
        ax = axes[idx]
        neigh_gdf.plot(ax=ax, facecolor="#f7f7f7", edgecolor="#999999", linewidth=0.4, alpha=0.6)
        vals = map_df[col].abs()
        vmax = max(vals.quantile(0.95), 1)
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        map_sorted = map_df.sort_values(col, key=abs, ascending=True)
        map_sorted.plot(column=col, ax=ax, legend=True, cmap="RdYlGn", norm=norm,
                        linewidth=3.0, alpha=0.85,
                        legend_kwds={"label": f"{label} ({unit})", "shrink": 0.4})
        top = pd.concat([map_df.nlargest(n_best, col), map_df.nsmallest(n_worst, col)]).drop_duplicates("route_short_name")
        placed = []
        for _, row in top.iterrows():
            pt = _best_label_point(row.geometry, placed)
            placed.append(pt)
            v = row[col]
            sign = "+" if v > 0 else ""
            tier = row.get("ptn_tier", "")
            tier_short = tier[:3].upper() if tier else ""
            ax.annotate(f"{row['route_short_name']} ({tier_short})\n{sign}{v:.0f} {unit}",
                        (pt.x, pt.y), fontsize=5.5, fontweight="bold", ha="center",
                        bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                                  edgecolor="gray", alpha=0.9))
        ax.set_xlim(route_bounds[0] - x_pad, route_bounds[2] + x_pad)
        ax.set_ylim(route_bounds[1] - y_pad, route_bounds[3] + y_pad)
        cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zoom=12)
        ax.set_title(label, fontsize=13, fontweight="bold")
        ax.set_axis_off()

    fig.subplots_adjust(top=0.97, bottom=0.02, left=0.01, right=0.99, wspace=0.05)
    save_report_figure(fig, "prepost_combined.png",
                       figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    plt.close(fig)


## 3b. Statistical Significance — Welch's t-test on Headway Change

In [ ]:
from scipy import stats
import numpy as np

if not route_comparison_table.empty and "headway_pre" in route_comparison_table.columns:
    # Drop rows where either headway is missing — must be paired (same route group matched pre/post)
    paired = route_comparison_table[["headway_pre", "headway_post", "ptn_tier"]].dropna(
        subset=["headway_pre", "headway_post"]
    )

    pre_headways  = paired["headway_pre"]
    post_headways = paired["headway_post"]

    # Paired t-test: each row is the same route group measured pre and post
    t_stat, p_value = stats.ttest_rel(pre_headways, post_headways)

    mean_pre  = pre_headways.mean()
    mean_post = post_headways.mean()
    diff      = (pre_headways - post_headways)
    cohens_d  = diff.mean() / diff.std()

    print("=== Paired t-test: Pre-PTN vs Post-PTN Headways (matched route groups) ===")
    print(f"Pre-PTN mean headway:  {mean_pre:.1f} min")
    print(f"Post-PTN mean headway: {mean_post:.1f} min")
    print(f"Mean improvement:      {diff.mean():.1f} min (n={len(paired)} matched routes)")
    print(f"t-statistic: {t_stat:.3f}")
    print(f"p-value:     {p_value:.4f}  ({'SIGNIFICANT' if p_value < 0.05 else 'not significant'} at α=0.05)")
    print(f"Cohen's d:   {cohens_d:.3f} ({'large' if abs(cohens_d) >= 0.8 else 'medium' if abs(cohens_d) >= 0.5 else 'small'} effect)")

    # Per-tier paired t-tests
    print("\n=== Per-tier Paired t-test ===")
    tier_results = []
    for tier in paired["ptn_tier"].dropna().unique():
        tier_df = paired[paired["ptn_tier"] == tier]
        if len(tier_df) >= 2:
            tt, pp = stats.ttest_rel(tier_df["headway_pre"], tier_df["headway_post"])
            tier_results.append({
                "ptn_tier": tier,
                "n_routes": len(tier_df),
                "mean_pre": tier_df["headway_pre"].mean().round(1),
                "mean_post": tier_df["headway_post"].mean().round(1),
                "mean_improvement": (tier_df["headway_pre"] - tier_df["headway_post"]).mean().round(1),
                "t_stat": round(tt, 3),
                "p_value": round(pp, 4),
                "significant": pp < 0.05,
            })
    display(pd.DataFrame(tier_results))
else:
    print("Headway comparison data not available.")

## 4. Speed and Trip Changes

In [ ]:
# (merged into prepost_combined)


## 5. Interpretation

Use the exported route comparison CSV for the report, dashboard, and final synthesis. Keep the route-change crosswalk under version control and update it manually when route renaming or splitting requires human judgment.

## 5. Employment Access Change

This section summarizes how neighbourhood access to employment-heavy destinations changed between the baseline feed and the current feed.


In [ ]:
# --- equity_triptych.png: 3-panel choropleth (modal share + immigrants + employment access) ---
import geopandas as gpd
import contextily as cx
from matplotlib.colors import TwoSlopeNorm

neigh_gdf = ctx.working_db.neighbourhood_gdf().to_crs(epsg=3857)
modal_share = coverage_analyzer.modal_share_by_neighbourhood()

# Census demographics
census_nb = ctx.working_db.query(
    "SELECT neighbourhood, pct_recent_immigrants, median_household_income_2020 "
    "FROM ywg_census_by_neighbourhood"
)

has_modal = not modal_share.empty
has_census = not census_nb.empty
has_jobs = not jobs_access_comparison_table.empty

if not has_modal and not has_census and not has_jobs:
    save_placeholder_figure("equity_triptych.png", "Equity data not available.",
                            figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
else:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Panel 1: Transit commute modal share
    ax1 = axes[0]
    if has_modal:
        gdf1 = neigh_gdf.merge(modal_share, on="neighbourhood", how="left")
        col = [c for c in gdf1.columns if 'transit' in c.lower() or 'modal' in c.lower() or 'pct_commute' in c.lower()]
        plot_col = col[0] if col else modal_share.columns[-1]
        gdf1.plot(column=plot_col, ax=ax1, legend=True, cmap="YlOrRd",
                  edgecolor="gray", linewidth=0.2, missing_kwds={"color": "lightgray"},
                  legend_kwds={"label": "% Transit Commuters", "shrink": 0.6})
        cx.add_basemap(ax1, source=cx.providers.CartoDB.Positron, zoom=12)
        ax1.set_title("Transit Commute Share\n(Census 2021)", fontsize=12, fontweight="bold")
    else:
        ax1.text(0.5, 0.5, "No modal share data", ha="center", va="center")
    ax1.set_axis_off()

    # Panel 2: Recent immigrant concentration
    ax2 = axes[1]
    if has_census:
        gdf2 = neigh_gdf.merge(census_nb, on="neighbourhood", how="left")
        gdf2.plot(column="pct_recent_immigrants", ax=ax2, legend=True, cmap="GnBu",
                  edgecolor="gray", linewidth=0.2, missing_kwds={"color": "lightgray"},
                  legend_kwds={"label": "% Recent Immigrants", "shrink": 0.6})
        cx.add_basemap(ax2, source=cx.providers.CartoDB.Positron, zoom=12)
        ax2.set_title("Recent Immigrant Concentration\n(Census 2016-2021)", fontsize=12, fontweight="bold")
    else:
        ax2.text(0.5, 0.5, "No census data", ha="center", va="center")
    ax2.set_axis_off()

    # Panel 3: Employment access change
    ax3 = axes[2]
    if has_jobs:
        gdf3 = neigh_gdf.merge(
            jobs_access_comparison_table[["neighbourhood", "jobs_access_change"]],
            on="neighbourhood", how="left",
        )
        vmax = gdf3["jobs_access_change"].abs().quantile(0.95)
        vmax = max(vmax, 10)
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        gdf3.plot(column="jobs_access_change", ax=ax3, legend=True, cmap="RdYlGn",
                  norm=norm, edgecolor="gray", linewidth=0.2,
                  missing_kwds={"color": "lightgray"},
                  legend_kwds={"label": "Jobs Access Change", "shrink": 0.6})
        cx.add_basemap(ax3, source=cx.providers.CartoDB.Positron, zoom=12)
        ax3.set_title("Employment Access Change\n(Pre-PTN vs Current)", fontsize=12, fontweight="bold")
    else:
        ax3.text(0.5, 0.5, "No jobs data", ha="center", va="center")
    ax3.set_axis_off()

    plt.tight_layout()
    save_report_figure(fig, "equity_triptych.png",
                       figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    plt.close(fig)




## 6. Census Validation Figures

In [ ]:
# (consolidated into equity_triptych above)


In [ ]:
# --- demand_validation.png: demand vs supply + commute duration (2 panels) ---
demand_supply = coverage_analyzer.departure_demand_vs_gtfs_supply()
commute_validation = coverage_analyzer.commute_duration_vs_r5py()

has_demand = not demand_supply.empty and "gtfs_departures" in demand_supply.columns
has_commute = not commute_validation.empty and "census_mean_commute_min" in commute_validation.columns

if not has_demand and not has_commute:
    save_placeholder_figure("demand_validation.png", "Demand/commute data not available.",
                            figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: demand vs supply
    if has_demand:
        ax1 = axes[0]
        ax1.bar(demand_supply["hour_label"], demand_supply["census_demand_pct"],
                color="#4292c6", alpha=0.7, label="Census departure demand (%)")
        ax1.set_xlabel("Departure Time Window", fontsize=11)
        ax1.set_ylabel("Census Demand (%)", color="#4292c6", fontsize=11)
        ax1.tick_params(axis="y", labelcolor="#4292c6")
        ax1.tick_params(axis="x", rotation=30)
        ax2 = ax1.twinx()
        ax2.plot(demand_supply["hour_label"], demand_supply["gtfs_departures"],
                 color="#d95f02", marker="o", linewidth=2.5, label="GTFS departures")
        ax2.set_ylabel("GTFS Scheduled Departures", color="#d95f02", fontsize=11)
        ax2.tick_params(axis="y", labelcolor="#d95f02")
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)
        ax1.set_title("Census Departure Demand vs GTFS Supply", fontsize=12, fontweight="bold")
    else:
        axes[0].text(0.5, 0.5, "No demand data", ha="center", va="center")

    # Right: commute duration
    if has_commute:
        ax = axes[1]
        ax.hist(commute_validation["census_mean_commute_min"].dropna(), bins=30,
                color="#4292c6", alpha=0.6, label="Census mean commute", density=True)
        has_r5py = "r5py_p50_travel_time_min" in commute_validation.columns and commute_validation["r5py_p50_travel_time_min"].notna().any()
        if has_r5py:
            ax.hist(commute_validation["r5py_p50_travel_time_min"].dropna(), bins=30,
                    color="#d95f02", alpha=0.6, label="r5py modelled travel time", density=True)
        ax.set_xlabel("Travel Time (minutes)", fontsize=11)
        ax.set_ylabel("Density", fontsize=11)
        ax.set_title("Census Commute Duration Distribution", fontsize=12, fontweight="bold")
        ax.legend(fontsize=9)
    else:
        axes[1].text(0.5, 0.5, "No commute data", ha="center", va="center")

    plt.tight_layout()
    save_report_figure(fig, "demand_validation.png",
                       figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    plt.close(fig)


In [ ]:
# (merged into demand_validation)


In [ ]:
# (consolidated into equity_triptych above)
